In [ ]:
# === Tutorial bootstrap: fetch utils/ + sample data if missing (for Colab blob links) ===
import os, sys, urllib.request

REPO   = "amirfar76/neurips25-valid-hparam-selection"
BRANCH = "main"
BASE   = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"

def ensure_utils():
    os.makedirs("utils", exist_ok=True)
    for fname in ["csvio.py", "testing.py"]:
        url = f"{BASE}/utils/{fname}"
        dst = os.path.join("utils", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)
    if "utils" not in sys.path:
        sys.path.append(os.path.abspath("utils"))

def ensure_data():
    os.makedirs("data", exist_ok=True)
    for fname in ["sample_binary_losses.csv", "sample_real_losses.csv"]:
        url = f"{BASE}/data/{fname}"
        dst = os.path.join("data", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)

ensure_utils()
ensure_data()
print("Bootstrap done: utils/ and data/ available.")


# D — Reliability-Graph PT (scaffold)

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv
from utils.testing import hoeffding_pval, weighted_bh

csv_path='data/sample_real_losses.csv'
alpha_risk=1.2
alpha_fdr=0.1

ids, L, cols = load_losses_csv(csv_path)
rows=[]
for i, hp in enumerate(ids):
    losses=L[i]
    rhat=float(losses.mean()); n=len(losses)
    p=hoeffding_pval(rhat, n, alpha_risk)
    rows.append({'hyperparam_id':hp,'pval':p})

df=pd.DataFrame(rows)

# Example prior weights (replace with graph-based weights from RG-PT)
w = np.ones(len(df))
w[0] = 2.0  # pretend node 0 has more prior reliability

df['selected']=weighted_bh(df['pval'].values, w, alpha=alpha_fdr)
df.sort_values(['selected','pval'])